## Criação do meu primeiro Catálogo

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS demo_catalog
COMMENT 'Catálogo de demonstração criado na Aula3 - Unity Catalog da Jornada de Dados'

## Criação do meu primeiro Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS demo_catalog.bronze
COMMENT 'Schema bronze para dados brutos e heterogêneos'

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS demo_catalog.silver
COMMENT 'Schema silver para dados transformados'

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS demo_catalog.gold
COMMENT 'Schema gold para dados que serão utilizados pelo negócio'

## Criação de uma tabela

In [0]:

%sql
CREATE TABLE demo_catalog.bronze.sales (
  sale_id INT,
  product STRING,
  quantity INT,
  price DOUBLE,
  sale_date DATE
)
USING DELTA
COMMENT 'Tabela de vendas brutas no formato Delta';

## Populando a tabela

In [0]:
%sql
INSERT INTO demo_catalog.bronze.sales (sale_id, product, quantity, price, sale_date)
VALUES
  -- Grupo 1: Notebook
  (1, 'Notebook Dell', 1, 5500.00, DATE'2025-10-01'),
  (2, 'Notebook Dell', 2, 5200.00, DATE'2025-10-02'),
  (3, 'Notebook Dell', 1, 5300.00, DATE'2025-10-03'),
  (4, 'Notebook Dell', 3, 5400.00, DATE'2025-10-04'),
  (5, 'Notebook Dell', 1, 5600.00, DATE'2025-10-05'),

  -- Grupo 2: Monitor
  (6, 'Monitor LG 27"', 2, 1200.00, DATE'2025-10-06'),
  (7, 'Monitor LG 27"', 1, 1150.00, DATE'2025-10-07'),
  (8, 'Monitor LG 27"', 3, 1180.00, DATE'2025-10-08'),
  (9, 'Monitor LG 27"', 2, 1220.00, DATE'2025-10-09'),
  (10, 'Monitor LG 27"', 1, 1250.00, DATE'2025-10-10'),

  -- Grupo 3: Headset
  (11, 'Headset HyperX', 2, 600.00, DATE'2025-10-11'),
  (12, 'Headset HyperX', 1, 590.00, DATE'2025-10-12'),
  (13, 'Headset HyperX', 3, 620.00, DATE'2025-10-13'),
  (14, 'Headset HyperX', 2, 610.00, DATE'2025-10-14'),
  (15, 'Headset HyperX', 1, 630.00, DATE'2025-10-15');

In [0]:
%sql
SELECT * 
FROM demo_catalog.bronze.sales

## Criação de uma view - Ordem lógica dos dados

In [0]:
%sql
CREATE OR REPLACE VIEW demo_catalog.bronze.vw_sales_summary AS
SELECT
  product,
  SUM(quantity) AS total_sold,
  ROUND(SUM(quantity * price), 2) AS revenue
FROM demo_catalog.bronze.sales
GROUP BY product;


In [0]:
%sql
SELECT * 
FROM demo_catalog.bronze.vw_sales_summary

## Criação de Volume para arquivos

In [0]:
%sql
CREATE VOLUME demo_catalog.bronze.raw_files
COMMENT 'Volume para arquivos brutos de ingestão inicial'

## Visualização de imagens no Databricks

In [0]:
from PIL import Image

path = '/Volumes/demo_catalog/bronze/raw_files/gato.png'
img = Image.open(path)
display(img)


In [0]:
print(img.format, img.size, img.mode)

In [0]:
%pip install torch torchvision

In [0]:
import torch
from torchvision import models, transforms
import urllib.request

# Carregar rótulos do ImageNet
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
imagenet_labels = urllib.request.urlopen(labels_url).read().decode("utf-8").splitlines()

# Pré-processamento padrão do ResNet
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Carregar imagem e preparar tensor
input_tensor = preprocess(Image.open(path).convert("RGB")).unsqueeze(0)

# Modelo pré-treinado
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.eval()

# Inferência
with torch.no_grad():
    logits = model(input_tensor)
    probs = torch.nn.functional.softmax(logits, dim=1)[0]

topk = torch.topk(probs, k=5)
top5 = [(imagenet_labels[idx], float(probs[idx])) for idx in topk.indices.tolist()]
top5

In [0]:
from pyspark.sql import Row
import json

prediction = Row(
    file_path = path,
    top1_label = top5[0][0],
    top1_prob  = top5[0][1],
    top5_json  = json.dumps(top5)
)

spark.createDataFrame([prediction]) \
     .write.mode("append") \
     .saveAsTable("demo_catalog.bronze.image_predictions")

In [0]:
%sql
select * 
from demo_catalog.bronze.image_predictions

## Leitura de arquivos CSV, JSON e PARQUET

In [0]:
# === Parquet ===
df_parquet = spark.read.parquet("/Volumes/demo_catalog/bronze/raw_files/dados.parquet")
print("🟢 PARQUET:")
df_parquet.show(truncate=False)

# === CSV ===
df_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/demo_catalog/bronze/raw_files/dados.csv")
)
print("🟢 CSV:")
df_csv.show(truncate=False)

# === JSON ===
df_json = spark.read.json("/Volumes/demo_catalog/bronze/raw_files/dados.json")
print("🟢 JSON:")
df_json.show(truncate=False)


In [0]:
print(type(df_parquet))

In [0]:

# Exemplo de SELECT * (todos os dados)
df_parquet.select("*").show()

# Exemplo de SELECT específico
df_parquet.select("id", "nome", "salario").show()

# Exemplo com filtro (WHERE)
df_parquet.filter(df_parquet.salario > 9000).select("nome", "salario").show()


In [0]:
# Exemplo de SELECT * (todos os dados)
df_parquet.select("*").display()

In [0]:
# Exemplo de SELECT específico
df_parquet.select("id", "nome", "salario").display()

In [0]:
# Exemplo com filtro (WHERE)
df_parquet.filter(df_parquet.salario > 9000).select("nome", "salario").display()

## Usando TempViews pra salvar os dados

In [0]:
# Ler Parquet
df_parquet = spark.read.parquet("/Volumes/demo_catalog/bronze/raw_files/dados.parquet")
df_parquet.createOrReplaceTempView("vw_parquet")

# Ler CSV
df_csv = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/demo_catalog/bronze/raw_files/dados.csv")
)
df_csv.createOrReplaceTempView("vw_csv")

# Ler JSON
df_json = spark.read.json("/Volumes/demo_catalog/bronze/raw_files/dados.json")
df_json.createOrReplaceTempView("vw_json")

In [0]:
%sql
select * from vw_json